# Condition Occurrence — Allscripts Sunrise (SCM)

OMOP CDM v5.4 `condition_occurrence` hydration using the standard SCM pattern:
- reset target tables
- build coded bronze source into a silver temp view
- merge silver
- insert missing `source_to_condition_occurrence` keys
- merge gold into `_exponent.omop_scm.condition_occurrence`

### Source Tables
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocdetail_bkp` — coded diagnoses (ICD/SNOMED)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur` — parent document (dates, provider, visit linkage)

### Target
- Silver: `_exponent.omop_silver.condition_occurrence`
- Gold: `_exponent.omop.condition_occurrence`

### Strategy
- JOIN detail → document on `ClientDocumentGUID = doc.GUID`
- Filter to rows with valid `CodingScheme` / `CodingSchemeCode` (ICD-10-CM, ICD-9-CM, SNOMED)
- Map source codes to OMOP `condition_source_concept_id` via `concept` table
- Resolve standard `condition_concept_id` via `concept_relationship` (Maps to)
- Date: `COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen)`
- `condition_type_concept_id = 32817` (EHR encounter record)

### Dependencies
- `source_to_person` mapping must be populated for Allscripts SCM patients
- OMOP vocabulary tables (`concept`, `concept_relationship`) must be loaded

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.condition_occurrence;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.condition_occurrence;
-- This notebook uses `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocdetail_bkp` and `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur`. It does not fail fast on zero coded rows; a zero-row source simply results in zero merged rows.


In [ ]:
from pyspark.sql import functions as F


In [ ]:
source = "allscripts_scm"
detail_table = "_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocdetail_bkp"
doc_table = "_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur"

print(f"Source system: {source}")
print(f"Detail table: {detail_table}")
print(f"Document table: {doc_table}")


In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.condition_occurrence;


In [ ]:
%sql
DELETE FROM _exponent.omop_silver.condition_occurrence
WHERE source_system = 'allscripts_scm';


In [ ]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_condition_occurrence
WHERE source_system = 'allscripts_scm';


In [ ]:
df_detail = spark.table(detail_table)
df_doc = spark.table(doc_table)

joined_df = df_detail.alias("detail").join(
    df_doc.alias("doc"),
    F.col("detail.ClientDocumentGUID") == F.col("doc.GUID"),
    "inner",
)

coded_ready_df = joined_df.filter(
    (F.col("detail.Active") == True)
    & (F.col("doc.Active") == True)
    & (F.col("doc.IsCanceled") == False)
    & F.col("detail.CodingScheme").isNotNull()
    & F.col("detail.CodingSchemeCode").isNotNull()
    & (F.trim(F.col("detail.CodingSchemeCode")) != "")
    & F.col("detail.ClientGUID").isNotNull()
    & F.coalesce(
        F.col("doc.AuthoredDtm"),
        F.col("doc.ServiceDtmUTC"),
        F.col("doc.Entered"),
        F.col("detail.CreatedWhen"),
    ).isNotNull()
)

print(f"Detail rows: {df_detail.count():,}")
print(f"Document rows: {df_doc.count():,}")
print(f"Joined rows: {joined_df.count():,}")
print(f"Coded rows after source filters: {coded_ready_df.count():,}")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW silver_condition_occurrence AS
WITH source_data AS (
    SELECT
        detail.GUID AS detail_guid,
        detail.ClientGUID,
        detail.CodingScheme,
        detail.CodingSchemeCode,
        detail.HeadingOne,
        detail.CreatedWhen AS detail_created,
        doc.AuthoredDtm,
        doc.ServiceDtmUTC,
        doc.Entered AS doc_entered,
        doc.AuthoredProviderGUID,
        doc.ClientVisitGUID,
        CASE
            WHEN UPPER(detail.CodingScheme) LIKE '%ICD-10%' OR UPPER(detail.CodingScheme) LIKE '%ICD10%' THEN 'ICD10CM'
            WHEN UPPER(detail.CodingScheme) LIKE '%ICD-9%' OR UPPER(detail.CodingScheme) LIKE '%ICD9%' THEN 'ICD9CM'
            WHEN UPPER(detail.CodingScheme) LIKE '%SNOMED%' THEN 'SNOMED'
            ELSE NULL
        END AS omop_vocabulary_id,
        COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen) AS condition_start_ts,
        CONCAT_WS(CHR(31), '{source}', 'dbo_cv3clientdocdetail_bkp', 'GUID', CAST(detail.GUID AS STRING)) AS condition_occurrence_source_value,
        CONCAT_WS(CHR(31), '{source}', 'cv3client', 'GUID', CAST(detail.ClientGUID AS STRING)) AS person_source_value,
        CASE
            WHEN doc.AuthoredProviderGUID IS NOT NULL
            THEN CONCAT_WS(CHR(31), '{source}', 'dbo_cv3careprovider', 'GUID', CAST(doc.AuthoredProviderGUID AS STRING))
            ELSE NULL
        END AS provider_source_value,
        CASE
            WHEN doc.ClientVisitGUID IS NOT NULL
            THEN CONCAT_WS(CHR(31), '{source}', 'dbo_cv3clientvisit', 'GUID', CAST(doc.ClientVisitGUID AS STRING))
            ELSE NULL
        END AS visit_occurrence_source_value
    FROM {detail_table} detail
    INNER JOIN {doc_table} doc
        ON detail.ClientDocumentGUID = doc.GUID
    WHERE detail.Active = TRUE
      AND doc.Active = TRUE
      AND doc.IsCanceled = FALSE
      AND detail.CodingScheme IS NOT NULL
      AND detail.CodingSchemeCode IS NOT NULL
      AND TRIM(detail.CodingSchemeCode) != ''
      AND detail.ClientGUID IS NOT NULL
      AND COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen) IS NOT NULL
      AND COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen) >= TIMESTAMP('1900-01-01')
      AND COALESCE(doc.AuthoredDtm, doc.ServiceDtmUTC, doc.Entered, detail.CreatedWhen) <= CURRENT_TIMESTAMP()
), mapped AS (
    SELECT
        sd.condition_occurrence_source_value,
        stp.person_id,
        COALESCE(std_concept.concept_id, 0) AS condition_concept_id,
        DATE(sd.condition_start_ts) AS condition_start_date,
        sd.condition_start_ts AS condition_start_datetime,
        CAST(NULL AS DATE) AS condition_end_date,
        CAST(NULL AS TIMESTAMP) AS condition_end_datetime,
        CAST(32817 AS BIGINT) AS condition_type_concept_id,
        CAST(0 AS BIGINT) AS condition_status_concept_id,
        CAST(NULL AS STRING) AS stop_reason,
        stpr.provider_id AS provider_id,
        COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
        CAST(NULL AS BIGINT) AS visit_detail_id,
        sd.visit_occurrence_source_value,
        sd.CodingSchemeCode AS condition_source_value,
        COALESCE(src_concept.concept_id, 0) AS condition_source_concept_id,
        sd.HeadingOne AS condition_status_source_value,
        '{source}' AS source_system,
        CURRENT_TIMESTAMP() AS last_mod_tsp,
        ROW_NUMBER() OVER (
            PARTITION BY sd.condition_occurrence_source_value
            ORDER BY sd.condition_start_ts DESC
        ) AS rn
    FROM source_data sd
    INNER JOIN `_exponent`.`omop_mapping`.`source_to_person` stp
      ON stp.person_source_value = sd.person_source_value
     AND stp.active_flag = TRUE
    LEFT JOIN `_exponent`.`omop_mapping`.`source_to_provider` stpr
      ON stpr.provider_source_value = sd.provider_source_value
     AND stpr.active_flag = TRUE
    LEFT JOIN `_exponent`.`omop_mapping`.`source_to_visit_occurrence` stvo
      ON stvo.visit_occurrence_source_value = sd.visit_occurrence_source_value
     AND stvo.source_system = '{source}'
     AND stvo.active_flag = TRUE
    LEFT JOIN `_exponent`.`omop_scm`.`visit_occurrence` vo
      ON vo.visit_source_value = sd.visit_occurrence_source_value
    LEFT JOIN `_exponent`.`omop`.`concept` src_concept
      ON src_concept.concept_code = sd.CodingSchemeCode
     AND src_concept.vocabulary_id = sd.omop_vocabulary_id
    LEFT JOIN `_exponent`.`omop`.`concept_relationship` cr
      ON cr.concept_id_1 = src_concept.concept_id
     AND cr.relationship_id = 'Maps to'
    LEFT JOIN `_exponent`.`omop`.`concept` std_concept
      ON std_concept.concept_id = cr.concept_id_2
     AND std_concept.standard_concept = 'S'
     AND std_concept.domain_id = 'Condition'
    WHERE sd.omop_vocabulary_id IS NOT NULL
)
SELECT
    condition_occurrence_source_value,
    person_id,
    condition_concept_id,
    condition_start_date,
    condition_start_datetime,
    condition_end_date,
    condition_end_datetime,
    condition_type_concept_id,
    condition_status_concept_id,
    stop_reason,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    visit_occurrence_source_value,
    condition_source_value,
    condition_source_concept_id,
    condition_status_source_value,
    source_system,
    last_mod_tsp
FROM mapped
WHERE rn = 1
""")

silver_count = spark.table("silver_condition_occurrence").count()
print(f"Silver staging rows: {silver_count:,}")
display(spark.table("silver_condition_occurrence").limit(25))


In [ ]:
# -- Merge to Silver layer --
spark.sql("""
MERGE INTO _exponent.omop_silver.condition_occurrence AS t
USING silver_condition_occurrence AS s
ON t.condition_occurrence_source_value = s.condition_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.condition_concept_id <=> s.condition_concept_id)
  OR NOT (t.condition_start_date <=> s.condition_start_date)
  OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
  OR NOT (t.condition_end_date <=> s.condition_end_date)
  OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
  OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
  OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.condition_source_value <=> s.condition_source_value)
  OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
  OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                     = s.person_id,
  t.condition_concept_id          = s.condition_concept_id,
  t.condition_start_date          = s.condition_start_date,
  t.condition_start_datetime      = s.condition_start_datetime,
  t.condition_end_date            = s.condition_end_date,
  t.condition_end_datetime        = s.condition_end_datetime,
  t.condition_type_concept_id     = s.condition_type_concept_id,
  t.condition_status_concept_id   = s.condition_status_concept_id,
  t.stop_reason                   = s.stop_reason,
  t.provider_id                   = s.provider_id,
  t.visit_occurrence_id           = s.visit_occurrence_id,
  t.visit_detail_id               = s.visit_detail_id,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.condition_source_value        = s.condition_source_value,
  t.condition_source_concept_id   = s.condition_source_concept_id,
  t.condition_status_source_value = s.condition_status_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  visit_occurrence_source_value,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.condition_occurrence_source_value,
  s.person_id,
  s.condition_concept_id,
  s.condition_start_date,
  s.condition_start_datetime,
  s.condition_end_date,
  s.condition_end_datetime,
  s.condition_type_concept_id,
  s.condition_status_concept_id,
  s.stop_reason,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.visit_occurrence_source_value,
  s.condition_source_value,
  s.condition_source_concept_id,
  s.condition_status_source_value,
  s.source_system,
  CURRENT_TIMESTAMP()
);
""")


In [ ]:
# -- Insert new mappings to source_to_condition_occurrence --
spark.sql("""
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.condition_occurrence_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, condition_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.condition_occurrence
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
  ON s.condition_occurrence_source_value = x.condition_occurrence_source_value
 AND s.source_system = x.source_system;
""")


In [ ]:
# -- Merge to Gold layer --
spark.sql("""
MERGE INTO _exponent.omop_scm.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    s.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    s.provider_id AS provider_id,
    COALESCE(s.visit_occurrence_id, stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    s.visit_detail_id AS visit_detail_id,
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.source_system = s.source_system
   AND sco.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = s.visit_occurrence_source_value
   AND stvo.source_system = s.source_system
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = s.visit_occurrence_source_value
  WHERE s.source_system = 'allscripts_scm'
    AND s.person_id IS NOT NULL
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
)
VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);
""")


In [ ]:
silver_rows = spark.sql("""
SELECT COUNT(*) AS silver_rows
FROM _exponent.omop_silver.condition_occurrence
WHERE source_system = 'allscripts_scm'
""")

gold_rows = spark.sql("""
SELECT COUNT(*) AS gold_rows
FROM _exponent.omop_scm.condition_occurrence
""")

print("SCM condition_occurrence counts after load")
display(silver_rows)
display(gold_rows)
display(
    spark.sql("""
    SELECT *
    FROM _exponent.omop_silver.condition_occurrence
    WHERE source_system = 'allscripts_scm'
    LIMIT 25
    """)
)
